# Christmasmodel — Colab training

All execution belongs in a **Google Colab GPU runtime**. This notebook has no saved outputs and has not been run by Codex.

The first cell fetches the code directly from the GitHub review branch and pins its exact commit in your Drive workspace. No manual source-folder upload is needed. Reconnecting reuses that commit. Choose a new versioned workspace to adopt later code changes.

Training text remains dated **1920 or earlier**. Choose the small scratch baseline first. The two SmolLM2 configurations form a matched scratch/pretrained pair; the pretrained branch's prior knowledge is not restricted to that period. Do not use an old project checkpoint with the new holdouts.


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab; do not execute it locally.") from exc

from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import quote
import json, os, re, subprocess, sys

drive.mount("/content/drive")
REPOSITORY = "wildfellhall/christmasmodel"
REF = "codex-colab-story-training"
WORKSPACE = Path("/content/drive/MyDrive/christmasmodel/experiments-v1")
WORKSPACE.mkdir(parents=True, exist_ok=True)
revision_file = WORKSPACE / "source_revision.json"
if revision_file.exists():
    revision = json.loads(revision_file.read_text())
    if revision["repository"] != REPOSITORY:
        raise ValueError("This workspace belongs to another repository.")
    SOURCE_SHA = revision["sha"]
else:
    if re.fullmatch(r"[0-9a-f]{40}", REF):
        SOURCE_SHA = REF
    else:
        request = Request(f"https://api.github.com/repos/{REPOSITORY}/commits/{quote(REF, safe='')}",
                          headers={"Accept": "application/vnd.github+json", "User-Agent": "christmasmodel-colab"})
        with urlopen(request, timeout=60) as response:
            SOURCE_SHA = json.load(response)["sha"]
    revision = {"repository": REPOSITORY, "requested_ref": REF, "sha": SOURCE_SHA}
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA):
    raise ValueError("Expected an immutable Git commit SHA.")
RUNTIME = Path("/content") / f"christmasmodel-{SOURCE_SHA[:12]}"
if not (RUNTIME / ".git").exists():
    subprocess.run(["git", "init", str(RUNTIME)], check=True)
current = subprocess.run(["git", "-C", str(RUNTIME), "rev-parse", "HEAD"], capture_output=True, text=True)
if current.returncode != 0:
    subprocess.run(["git", "-C", str(RUNTIME), "fetch", "--depth", "1",
                    f"https://github.com/{REPOSITORY}.git", SOURCE_SHA], check=True)
    subprocess.run(["git", "-C", str(RUNTIME), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual = subprocess.check_output(["git", "-C", str(RUNTIME), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_SHA:
    raise ValueError("The Colab checkout differs from the pinned code revision.")
status = subprocess.check_output(["git", "-C", str(RUNTIME), "status", "--porcelain", "--untracked-files=no"], text=True)
if status.strip():
    raise ValueError("The Colab source checkout has edits; use a clean runtime for a reproducible run.")
temporary = revision_file.with_suffix(".tmp")
temporary.write_text(json.dumps(revision, indent=2) + "\n")
temporary.replace(revision_file)
os.chdir(RUNTIME)
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
print("Pinned code:", SOURCE_SHA, "\nRuntime:", RUNTIME, "\nPersistent workspace:", WORKSPACE)


## Install and verify in Colab

Run these cells before preparing the full corpus. The test suite includes small optimization and exact-resume checks. These execute inside Colab. If installation asks for a session restart, restart and rerun setup before continuing.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
                "pytest>=8,<9", "pandas>=2.2,<3", "matplotlib>=3.8,<4"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > GPU.")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Prepare frozen story data

The first run downloads the source books into Drive. Later runs verify and reuse those files. Story boundaries remove known front/back matter; unresolved anthologies are omitted and listed for further curation. Reprints and overlapping source works stay in the same split. The split contains Christmas, winter, and broader historical fiction. Nothing evaluates the test split here.

Changing source rules requires a new versioned `WORKSPACE`; do not overwrite an existing corpus.


In [ ]:
from colab_workflow import prepare, configuration, run
manifest = prepare(WORKSPACE, RUNTIME)
print(json.dumps(manifest["counts"], indent=2))
catalog = json.loads((WORKSPACE / "story_catalog.json").read_text())
print("Duplicate units omitted:", len(manifest["duplicates"]))
print("Anthologies awaiting boundary review:", len(catalog["pending_anthologies"]))
print("Historical cutoff:", manifest["historical_cutoff_inclusive"])


## Choose a branch and budget

- `story-scratch`: small GPT-2, random weights, tokenizer learned only from training books.
- `story-matched-scratch`: random SmolLM2-135M weights, native tokenizer/configuration.
- `story-pretrained`: pinned SmolLM2-135M weights, same tokenizer/configuration as the matched scratch branch.

Use the same corpus, target budget, context, and seeds for the matched pair. Each branch has its own learning rate. The larger scratch model is a comparison, not an assumption that size solves the limited-data problem. No full run starts until you enable the training cell. Checkpoint/review files can occupy several GB; allow Drive space for each branch.

To change a frozen run's settings, use a new `RUN_TAG`. Rerunning an unchanged configuration resumes its latest complete optimizer checkpoint.


In [ ]:
VARIANT = "story-scratch" # @param ["story-scratch", "story-matched-scratch", "story-pretrained"]
RUN_TAG = "v1" # @param {type:"string"}
TOKEN_BUDGET = 32000000 # @param {type:"integer"}
CONTEXT = 512 # @param [512, 1024] {type:"raw"}
EVAL_STEPS = 500 # @param {type:"integer"}
GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}

settings = json.loads((RUNTIME / "configs" / f"{VARIANT}.json").read_text())
settings.update(token_budget=TOKEN_BUDGET, context=CONTEXT, lookback=None,
                eval_steps=EVAL_STEPS, gradient_checkpointing=GRADIENT_CHECKPOINTING,
                output=f"runs/{VARIANT}-{RUN_TAG}")
CONFIG = configuration(settings, WORKSPACE)
print(CONFIG.read_text())


In [ ]:
RUN_SMOKE = True # @param {type:"boolean"}
if RUN_SMOKE:
    smoke_config = configuration(settings, WORKSPACE, smoke=True)
    print(json.dumps(run(smoke_config), indent=2))
    import gc
    gc.collect()
    torch.cuda.empty_cache()
# Smoke checkpoints cannot be promoted for generation or used for final test evaluation.


In [ ]:
RUN_TRAINING = False # @param {type:"boolean"}
if RUN_TRAINING:
    summary = run(CONFIG)
    print(json.dumps(summary, indent=2))
else:
    print("Enable RUN_TRAINING when ready. No full training run started.")


## Inspect raw prose and select a checkpoint

`best_nll.json` identifies the lowest validation loss; `last.json` identifies the last update. Neither becomes the generator automatically. Review the saved CSV for the checkpoint you want to use. Every prompt has greedy generation and two sampled seeds. Scores are **0 = failure, 1 = mixed, 2 = clear success** for grammar, continuity, progression, nonrepetition, and ending. An unfinished continuation does not have to resolve the entire story.

Keep every raw failure and all prompt/continuation text unchanged. Edit the CSV in Drive, or enable the upload cell after editing a downloaded copy. Promotion checks the full panel, per-pool pass rates, provenance, and the validation-loss allowance.


In [ ]:
REVIEW_STEP = "best-nll" # @param {type:"string"}
cfg = json.loads(CONFIG.read_text())
RUN = Path(cfg["output"])
CHECKPOINT = RATINGS = None
if not (RUN / "best_nll.json").exists():
    print("No completed training review yet. Run the training cell before selecting a checkpoint.")
else:
    if REVIEW_STEP == "best-nll":
        pointer = json.loads((RUN / "best_nll.json").read_text())
        CHECKPOINT = RUN / pointer["checkpoint"]
    elif REVIEW_STEP == "0":
        CHECKPOINT = RUN / "initial" / "model"
    else:
        CHECKPOINT = RUN / "reviews" / f"step-{int(REVIEW_STEP)}" / "model"
    RATINGS = CHECKPOINT.parent / "ratings.csv"
    report = json.loads((CHECKPOINT.parent / "review.json").read_text())
    print("Checkpoint:", CHECKPOINT)
    print("Ratings:", RATINGS)
    print(json.dumps(report["metrics"], indent=2))
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(RATINGS).head(10))
    history = []
    paths = [RUN / "initial" / "review.json", *sorted((RUN / "reviews").glob("step-*/review.json"))]
    for path in paths:
        if path.exists():
            item = json.loads(path.read_text())
            metrics = item["metrics"][str(cfg["context"])]["all"]
            history.append({"step": item["step"], "nll": metrics["nll"], "bits_per_byte": metrics["bits_per_byte"]})
    frame = pd.DataFrame(history).sort_values("step")
    display(frame)
    if len(frame) > 1:
        frame.plot(x="step", y=["nll", "bits_per_byte"], subplots=True, title="Validation metrics — inspect prose separately")


In [ ]:
UPLOAD_RATINGS = False # @param {type:"boolean"}
if UPLOAD_RATINGS:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one edited ratings CSV.")
    RATINGS = WORKSPACE / "ratings" / f"{RUN.name}-{CHECKPOINT.parent.name}.csv"
    RATINGS.parent.mkdir(parents=True, exist_ok=True)
    RATINGS.write_bytes(next(iter(uploaded.values())))
    print("Uploaded:", RATINGS)


In [ ]:
PROMOTE = False # @param {type:"boolean"}
REVIEWER = "" # @param {type:"string"}
if PROMOTE:
    from story_review import select_checkpoint
    selection = select_checkpoint(RUN, CHECKPOINT, RATINGS, REVIEWER,
                                  min_pass_rate=0.8, max_nll_regression=0.15)
    print(json.dumps(selection, indent=2))


## Generate with the reviewed selection

The default repetition penalty is 1.0. Try other decoding values explicitly and review raw text; decoding controls cannot establish that training produced a coherent model.


In [ ]:
GENERATE = False # @param {type:"boolean"}
USE_REVIEWED_SELECTION = True # @param {type:"boolean"}
PROMPT = "The snow had covered the path to the cottage, but Mary could still see a light in the window." # @param {type:"string"}
if GENERATE:
    if not USE_REVIEWED_SELECTION and CHECKPOINT is None:
        raise ValueError("Choose a saved checkpoint in the review cell first.")
    source_args = ["--run", str(RUN)] if USE_REVIEWED_SELECTION else ["--checkpoint", str(CHECKPOINT)]
    subprocess.run([sys.executable, "model.py", "stories", "generate", *source_args,
                    "--prompt", PROMPT, "--tokens", "192", "--repetition-penalty", "1.0"], check=True)


## Optional controlled comparisons

Generate configs without launching them. Start with CE; change one objective, pool share, author balance, context, or learning rate at a time. Reviewed preferences require a separately annotated training-only JSONL file described in the README. Source-versus-arbitrary-rollout preferences and readability weighting are absent from this workflow.

For blind comparison, give paths to two or more completed `review.json` files with the same text, prompts, modes, and decoding. Rate the shuffled CSV before opening the unblinding map. Across tokenizers, compare **bits per byte on identical text and context**, not token NLL.


In [ ]:
MAKE_ABLATIONS = False # @param {type:"boolean"}
if MAKE_ABLATIONS:
    from story_experiments import ablation_configs
    print(json.dumps(ablation_configs(CONFIG, WORKSPACE / f"{RUN.name}-ablations"), indent=2))

MAKE_BLIND_PANEL = False # @param {type:"boolean"}
REVIEW_PATHS = []  # e.g. [str(scratch_review), str(pretrained_review)]
COMPARISON = WORKSPACE / "comparison-v1"
if MAKE_BLIND_PANEL:
    from story_experiments import compare_reviews
    print(json.dumps(compare_reviews(REVIEW_PATHS, COMPARISON), indent=2))
# After rating: story_experiments.unblind_ratings(COMPARISON, edited_csv, output_folder)
# The returned per-branch CSVs can be supplied to select_checkpoint.


## Export and verify in Colab

This uses the reviewed selection. Hugging Face/GGUF export supports the small GPT-2 and the Llama architecture used by SmolLM2. Verification compares tokenizer probes, next-token predictions, and multiple greedy continuations against the pinned llama.cpp runtime. Lower-precision exports can diverge; inspect the verification report and prose. No upload or publication is performed.


In [ ]:
EXPORT = False # @param {type:"boolean"}
if EXPORT:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-export.txt"], check=True)
    subprocess.run([sys.executable, "export.py", "--model-dir", str(RUN),
                    "--output-dir", str(WORKSPACE / "exports" / RUN.name),
                    "--llama-dir", "/content/llama.cpp", "--setup", "--outtype", "f16", "--verify"], check=True)


## Final held-out test (only after choosing the model)

Keep this disabled while adjusting models and decoding. The test set is held out from this project's training; unknown overlap with a pretrained base's original corpus remains possible. This cell writes an explicit final report and never feeds its result back into training.


In [ ]:
RUN_FINAL_TEST = False # @param {type:"boolean"}
if RUN_FINAL_TEST:
    subprocess.run([sys.executable, "model.py", "stories", "test", "--run", str(RUN),
                    "--output", str(WORKSPACE / f"{RUN.name}-final-test.json")], check=True)
